# Sale Forecasting -- LightGBM Rolling-Sample (T1-T10 -> T11, 2025)

## Tiêu đề và mục tiêu bài toán

**Bài toán**: Dự báo doanh số bán hàng (Sale Forecasting) theo từng cặp `(location, item_id)`.

**Mô tả**:
- Dữ liệu giao dịch mua hàng (event "purchased") năm 2025.
- Dữ liệu event khác (view_item, add-to-cart) có thể bổ sung sau; notebook này chỉ dùng purchase.
- **Train data**: 2025, **Blind test**: tháng 1/2026.
- Output submission: bảng 3 cột `location`, `item_id`, `prediction`.
- Đánh giá trên các location có giao dịch, loại bỏ item có `sale_status = 0`.

**Metrics**:
| Metric | Mô tả |
|---|---|
| MAE quantity | Mean Absolute Error trên số lượng bán |
| **MAPE quantity** | **Mean Absolute Percentage Error trên số lượng (metric chính)** |
| MAE revenue | Mean Absolute Error trên doanh thu |
| MAPE revenue | Mean Absolute Percentage Error trên doanh thu |

## Chiến lược Rolling-Sample Training

Thay vì chia ngẫu nhiên 80/20, notebook này sử dụng **rolling temporal samples** để tạo dữ liệu huấn luyện.

**Cách tạo rolling samples:**
- Sample 1: X từ T1-T6 -> y T7
- Sample 2: X từ T1-T7 -> y T8
- Sample 3: X từ T1-T8 -> y T9
- Sample 4: X từ T1-T9 -> y T10

Tất cả 4 sample được ghép lại thành một bảng `train_df` duy nhất để huấn luyện **một** mô hình LightGBM.

**Validation (tháng 11):**
- X_val từ T1-T10 -> y_val T11
- Tháng 11 **không** được dùng trong huấn luyện.

**Ưu điểm của rolling samples:**
- Tạo nhiều dòng huấn luyện hơn, không chỉ nhiều features.
- Mô hình học từ nhiều ví dụ thời gian khác nhau (temporal examples).
- Phù hợp với bài toán dự báo hơn là chia ngẫu nhiên.

**Quy tắc chống rò rỉ dữ liệu (Anti-leakage):**
- Với mỗi tháng mục tiêu `t`, features chỉ được tính từ các tháng **trước** `t`.
- VD: Mục tiêu T7 -> features chỉ từ T1-T6; mục tiêu T11 -> features chỉ từ T1-T10.

In [ ]:
# Cài đặt thư viện cần thiết (chạy 1 lần)
import subprocess, sys
for pkg in ["lightgbm","polars","pyarrow","pandas","numpy","scikit-learn","matplotlib"]:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Da cai dat xong cac thu vien.")

Da cai dat xong cac thu vien.


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import warnings, os, pathlib
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

ROOT = pathlib.Path(r"C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website")
RAW = ROOT / "raw_data"
OUT = ROOT / "Additional Doc" / "Đồ-Án"
OUT.mkdir(parents=True, exist_ok=True)

TXN_PATH = RAW / "transaction_full_2025.parquet"
ITEMS_PATH = RAW / "items.parquet"

print(f"Transaction file: {TXN_PATH}")
print(f"Items file: {ITEMS_PATH}")
print(f"Output folder: {OUT}")

Transaction file: C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website\raw_data\transaction_full_2025.parquet
Items file: C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website\raw_data\items.parquet
Output folder: C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website\Additional Doc\Đồ-Án


## Đọc dữ liệu và kiểm tra schema

In [ ]:
# Đọc items metadata
items_df = pl.read_parquet(str(ITEMS_PATH))
print("Items schema:")
print(items_df.schema)
print(f"   Shape: {items_df.shape}")
print(f"   sale_status phân bố: {items_df['sale_status'].value_counts().sort('sale_status')}")

Items schema:
Schema({'item_id': String, 'price': Decimal(precision=38, scale=4), 'category_l1': String, 'category_l2': String, 'category_l3': String, 'category': String, 'brand': String, 'manufacturer': String, 'description': String, 'sale_status': Int32, 'size': String})
   Shape: (29823, 11)
   sale_status phân bố: shape: (2, 2)
┌─────────────┬───────┐
│ sale_status ┆ count │
│ ---         ┆ ---   │
│ i32         ┆ u32   │
╞═════════════╪═══════╡
│ 0           ┆ 22973 │
│ 1           ┆ 6850  │
└─────────────┴───────┘


In [ ]:
txn_lazy = pl.scan_parquet(str(TXN_PATH))
print("Transaction schema:")
print(txn_lazy.schema)
txn_lazy.head(5).collect().to_pandas()

Transaction schema:
Schema({'bill_id': Int32, 'customer_id': Int32, 'item_id': String, 'price': Decimal(precision=38, scale=4), 'quantity': Int32, 'event_type': String, 'updated_date': Datetime(time_unit='us', time_zone=None), 'location_name': String})


,bill_id,customer_id,item_id,price,quantity,event_type,updated_date,location_name
0,137262467,6291331,6767000000002,285000.0000,1,Purchase,2025-04-12 12:52:41.910,QNG - 255 Quốc Lộ 1A
1,138193104,5505362,2265000000027,195000.0000,1,Purchase,2025-04-24 14:57:00.773,HCM - 88/4 Nguyễn Ảnh Thủ
2,138230425,7694873,6497000000004,312550.0000,1,Purchase,2025-04-24 20:35:40.027,BDU - 272 Đại Lộ Bình Dương
3,138235471,5837076,6587000000001,149000.0000,1,Purchase,2025-04-24 21:12:32.070,HCM - 703 Lê Hồng Phong
4,138239415,6781756,3880000000002,79000.0000,1,Purchase,2025-04-25 08:00:46.293,LDO - Quốc Lộ 20


## Tiền xử lý dữ liệu

**Các bước**:
1. Lọc chỉ event "Purchase" (case-insensitive).
2. Chuyển `updated_date` sang datetime, tạo cột `date` và `month`.
3. Tính `revenue = price * quantity`.
4. Loại bỏ tháng 12.
5. Join với items có `sale_status != 0`.
6. Đổi tên `location_name` -> `location`.

In [ ]:
txn_processed = (
    txn_lazy
    .filter(pl.col("event_type").str.to_lowercase().str.contains("purchase"))
    .with_columns(pl.col("updated_date").cast(pl.Datetime))
    .with_columns([
        pl.col("updated_date").dt.date().alias("date"),
        pl.col("updated_date").dt.month().alias("month"),
        pl.col("updated_date").dt.year().alias("year"),
        (pl.col("price").cast(pl.Float64) * pl.col("quantity").cast(pl.Float64)).alias("revenue"),
    ])
    .filter((pl.col("month") < 12) & (pl.col("year") == 2025))
    .rename({"location_name": "location"})
    .with_columns(pl.col("item_id").cast(pl.Utf8))
)

items_active = (
    items_df
    .filter(pl.col("sale_status") != 0)
    .select("item_id", "price", "category_l1", "category_l2", "category_l3",
            "category", "brand", "manufacturer", "size")
    .with_columns(pl.col("item_id").cast(pl.Utf8))
    .rename({"price": "item_price"})
)
print(f"Items active (sale_status != 0): {items_active.shape[0]:,} items")

txn_clean = (
    txn_processed
    .join(items_active.lazy(), on="item_id", how="inner")
    .collect()
)
print(f"Transactions sau tiền xử lý: {txn_clean.shape[0]:,} dòng")
print(f"   Phạm vi thời gian: {txn_clean['date'].min()} -> {txn_clean['date'].max()}")
print(f"   Số tháng: {txn_clean['month'].unique().sort()}")
print(f"   Số location: {txn_clean['location'].n_unique()}")
print(f"   Số item_id: {txn_clean['item_id'].n_unique()}")

Items active (sale_status != 0): 6,850 items
Transactions sau tiền xử lý: 32,142,167 dòng
   Phạm vi thời gian: 2025-01-01 -> 2025-11-30
   Số tháng: shape: (11,)
Series: 'month' [i8]
[
	1
	2
	3
	4
	5
	…
	7
	8
	9
	10
	11
]
   Số location: 987
   Số item_id: 5662


In [ ]:
# Tổng hợp theo (location, item_id, month)
monthly = (
    txn_clean.lazy()
    .group_by(["location", "item_id", "month"])
    .agg([
        pl.col("quantity").sum().alias("qty"),
        pl.col("revenue").sum().alias("rev"),
        pl.col("date").n_unique().alias("active_days"),
        pl.col("bill_id").n_unique().alias("num_bills"),
        pl.col("customer_id").n_unique().alias("num_customers"),
        pl.col("price").cast(pl.Float64).mean().alias("avg_price"),
        pl.col("price").cast(pl.Float64).last().alias("last_price"),
    ])
    .collect()
)
print(f"Bảng monthly: {monthly.shape[0]:,} dòng, {monthly.shape[1]} cột")
monthly.head(5).to_pandas()

Bảng monthly: 8,696,539 dòng, 10 cột


## Feature Engineering -- hàm `build_features`

Hàm `build_features(monthly, items_active, history_end_month, target_month)` tạo một bảng supervised dataset:
- Features tính từ tháng 1 đến `history_end_month`.
- Target là tổng số lượng/doanh thu của `target_month`.
- Mỗi dòng là một bộ `(location, item_id, target_month)`.

Quy tắc chống rò rỉ: features chỉ dùng dữ liệu **trước** tháng mục tiêu.

In [ ]:
def build_features(monthly_all, items_act, history_end_month, target_month):
    """Tao bang supervised dataset. Features chi dung du lieu thang 1 -> history_end_month."""
    monthly_hist = monthly_all.filter(pl.col("month") <= history_end_month)
    monthly_target = monthly_all.filter(pl.col("month") == target_month)

    skeleton_hist = monthly_hist.select("location", "item_id").unique()
    skeleton_tgt = monthly_target.select("location", "item_id").unique()
    skeleton = pl.concat([skeleton_hist, skeleton_tgt]).unique()

    # Qty pivot
    qty_pivot = (monthly_hist.lazy().select("location","item_id","month","qty").collect()
        .pivot(on="month", index=["location","item_id"], values="qty").fill_null(0))
    rename_q = {str(m): f"qty_m{m:02d}" for m in range(1, history_end_month+1)}
    qty_pivot = qty_pivot.rename({k:v for k,v in rename_q.items() if k in qty_pivot.columns})
    for m in range(1, 11):
        col = f"qty_m{m:02d}"
        if col not in qty_pivot.columns:
            qty_pivot = qty_pivot.with_columns(pl.lit(0).alias(col))

    # Rev pivot
    rev_pivot = (monthly_hist.lazy().select("location","item_id","month","rev").collect()
        .pivot(on="month", index=["location","item_id"], values="rev").fill_null(0))
    rename_r = {str(m): f"rev_m{m:02d}" for m in range(1, history_end_month+1)}
    rev_pivot = rev_pivot.rename({k:v for k,v in rename_r.items() if k in rev_pivot.columns})
    for m in range(1, 11):
        col = f"rev_m{m:02d}"
        if col not in rev_pivot.columns:
            rev_pivot = rev_pivot.with_columns(pl.lit(0.0).alias(col))

    features = skeleton.clone()
    features = features.join(qty_pivot, on=["location","item_id"], how="left").fill_null(0)
    rev_only = [c for c in rev_pivot.columns if c.startswith("rev_m")]
    features = features.join(rev_pivot.select(["location","item_id"]+rev_only), on=["location","item_id"], how="left").fill_null(0)

    hem = history_end_month

    # Recent qty
    features = features.with_columns([
        pl.col(f"qty_m{hem:02d}").alias("qty_last_1m"),
        (pl.col(f"qty_m{max(hem-1,1):02d}").alias("qty_last_2m") if hem >= 2 else pl.lit(0).alias("qty_last_2m")),
        (pl.col(f"qty_m{max(hem-2,1):02d}").alias("qty_last_3m") if hem >= 3 else pl.lit(0).alias("qty_last_3m")),
    ])

    # Rolling
    l3q = [f"qty_m{max(hem-i,1):02d}" for i in range(3)] if hem>=3 else [f"qty_m{m:02d}" for m in range(1,hem+1)]
    l6q = [f"qty_m{max(hem-i,1):02d}" for i in range(6)] if hem>=6 else [f"qty_m{m:02d}" for m in range(1,hem+1)]
    l3r = [f"rev_m{max(hem-i,1):02d}" for i in range(3)] if hem>=3 else [f"rev_m{m:02d}" for m in range(1,hem+1)]
    l6r = [f"rev_m{max(hem-i,1):02d}" for i in range(6)] if hem>=6 else [f"rev_m{m:02d}" for m in range(1,hem+1)]
    features = features.with_columns([
        (pl.sum_horizontal(*l3q)/len(l3q)).alias("qty_rolling_3m"),
        (pl.sum_horizontal(*l6q)/len(l6q)).alias("qty_rolling_6m"),
        (pl.sum_horizontal(*l3r)/len(l3r)).alias("rev_rolling_3m"),
        (pl.sum_horizontal(*l6r)/len(l6r)).alias("rev_rolling_6m"),
    ])

    # Aggregate
    aq = [f"qty_m{m:02d}" for m in range(1, hem+1)]
    ar = [f"rev_m{m:02d}" for m in range(1, hem+1)]
    nh = len(aq)
    features = features.with_columns([
        pl.sum_horizontal(aq).alias("qty_total_hist"),
        (pl.sum_horizontal(aq)/float(nh)).alias("qty_mean_hist"),
        pl.concat_list(aq).list.eval(pl.element().std()).list.first().alias("qty_std_hist"),
        pl.max_horizontal(aq).alias("qty_max_hist"),
        pl.sum_horizontal(ar).alias("rev_total_hist"),
        (pl.sum_horizontal(ar)/float(nh)).alias("rev_mean_hist"),
        pl.concat_list(ar).list.eval(pl.element().std()).list.first().alias("rev_std_hist"),
        pl.max_horizontal(ar).alias("rev_max_hist"),
    ])

    # Activity
    features = features.with_columns([pl.sum_horizontal([pl.col(c)>0 for c in aq]).alias("months_active_hist")])
    features = features.with_columns([(nh - pl.col("months_active_hist")).alias("zero_months_hist")])

    act = (monthly_hist.lazy().group_by(["location","item_id"]).agg([
        pl.col("active_days").sum().alias("active_days_hist"),
        pl.col("num_bills").sum().alias("num_bills_hist"),
        pl.col("num_customers").sum().alias("num_customers_hist"),
        pl.col("avg_price").mean().alias("avg_price_hist"),
        pl.col("last_price").last().alias("last_known_price"),
    ]).collect())
    features = features.join(act, on=["location","item_id"], how="left")

    # Price
    features = features.join(items_act.select("item_id", pl.col("item_price").cast(pl.Float64).alias("price_ref")), on="item_id", how="left")
    features = features.with_columns([pl.coalesce("last_known_price","avg_price_hist","price_ref").alias("price_ref_final")])

    # Trend
    if hem >= 2:
        features = features.with_columns([(pl.col(f"qty_m{hem:02d}")-pl.col(f"qty_m{hem-1:02d}")).alias("trend_qty_last_1_2")])
    else:
        features = features.with_columns(pl.lit(0).alias("trend_qty_last_1_2"))
    features = features.with_columns([(pl.col("qty_rolling_3m")-pl.col("qty_rolling_6m")).alias("trend_qty_r3_r6")])

    # Metadata
    mc = ["item_id","category_l1","category_l2","category_l3","category","brand","manufacturer","size"]
    features = features.join(items_act.select([c for c in mc if c in items_act.columns]), on="item_id", how="left")

    # Global
    gi = monthly_hist.lazy().group_by("item_id").agg([pl.col("qty").sum().alias("item_global_qty_hist"),pl.col("rev").sum().alias("item_global_rev_hist")]).collect()
    gl = monthly_hist.lazy().group_by("location").agg([pl.col("qty").sum().alias("loc_global_qty_hist"),pl.col("rev").sum().alias("loc_global_rev_hist")]).collect()
    features = features.join(gi, on="item_id", how="left").fill_null(0)
    features = features.join(gl, on="location", how="left").fill_null(0)

    # Temporal context
    features = features.with_columns([
        pl.lit(target_month).cast(pl.Int32).alias("target_month"),
        pl.lit(history_end_month).cast(pl.Int32).alias("history_end_month"),
        pl.lit(nh).cast(pl.Int32).alias("history_months_count"),
    ])

    # Target
    labels = monthly_target.lazy().group_by(["location","item_id"]).agg([pl.col("qty").sum().alias("y_qty"),pl.col("rev").sum().alias("y_revenue")]).collect()
    features = features.join(labels, on=["location","item_id"], how="left")
    features = features.with_columns([pl.col("y_qty").fill_null(0).alias("y_qty"), pl.col("y_revenue").fill_null(0.0).alias("y_revenue")])

    if "price_ref" in features.columns and "price_ref_final" in features.columns:
        features = features.drop("price_ref").rename({"price_ref_final": "price_ref"})

    return features

print("Ham build_features da duoc dinh nghia.")

## Tạo bảng huấn luyện Rolling-Sample

Ghép 4 sample:
- `build_features(..., history_end_month=6, target_month=7)`
- `build_features(..., history_end_month=7, target_month=8)`
- `build_features(..., history_end_month=8, target_month=9)`
- `build_features(..., history_end_month=9, target_month=10)`

Tháng 11 **không** được dùng trong huấn luyện.


In [ ]:
# === TAO ROLLING TRAINING DATA ===
rolling_configs = [(6, 7), (7, 8), (8, 9), (9, 10)]

train_parts = []
for hem, tm in rolling_configs:
    print(f"  Dang tao: history T1-T{hem} -> target T{tm} ...")
    part = build_features(monthly, items_active, hem, tm)
    print(f"    -> {part.shape[0]:,} dong, {part.shape[1]} cot")
    train_parts.append(part)

# Dam bao tat ca parts co cung cac cot
all_cols = train_parts[0].columns
for i, p in enumerate(train_parts):
    for c in all_cols:
        if c not in p.columns:
            train_parts[i] = p.with_columns(pl.lit(0).alias(c))
    train_parts[i] = train_parts[i].select(all_cols)

train_df = pl.concat(train_parts)
print(f"\nBang train tong (rolling): {train_df.shape[0]:,} dong, {train_df.shape[1]} cot")
print(f"  Phan bo target_month: {train_df['target_month'].value_counts().sort('target_month')}")


## Tạo bảng validation (T1-T10 -> T11)

Tháng 11 chỉ được sử dụng làm target để đánh giá, **không** dùng trong features.


In [ ]:
print("Dang tao: history T1-T10 -> target T11 ...")
val_df = build_features(monthly, items_active, history_end_month=10, target_month=11)
print(f"Bang validation: {val_df.shape[0]:,} dong, {val_df.shape[1]} cot")
print(f"  Co y_qty > 0: {val_df.filter(pl.col('y_qty') > 0).shape[0]:,}")
print(f"  Co y_qty = 0: {val_df.filter(pl.col('y_qty') == 0).shape[0]:,}")

for c in train_df.columns:
    if c not in val_df.columns:
        val_df = val_df.with_columns(pl.lit(0).alias(c))
val_df = val_df.select(train_df.columns)


## Chuẩn bị dữ liệu và huấn luyện LightGBM

In [ ]:
# Xac dinh cot features va target
id_cols = ["location", "item_id"]
target_col = "y_qty"
target_rev = "y_revenue"
drop_cols = id_cols + [target_col, target_rev]

train_pd = train_df.to_pandas()
val_pd = val_df.to_pandas()

feature_cols = [c for c in train_pd.columns if c not in drop_cols]
cat_cols = ["category_l1","category_l2","category_l3","category","brand","manufacturer","size"]
cat_cols = [c for c in cat_cols if c in feature_cols]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Feature cols: {len(feature_cols)} ({len(num_cols)} numeric + {len(cat_cols)} categorical)")


In [ ]:
# Encode categorical va fill NaN
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=int)
train_pd[cat_cols] = enc.fit_transform(train_pd[cat_cols].astype(str).fillna("__NA__"))
val_pd[cat_cols] = enc.transform(val_pd[cat_cols].astype(str).fillna("__NA__"))

train_pd[num_cols] = train_pd[num_cols].fillna(0)
val_pd[num_cols] = val_pd[num_cols].fillna(0)

X_train = train_pd[feature_cols]
y_train = train_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]
y_val_rev = val_pd[target_rev]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")


In [ ]:
# === TRAIN LIGHTGBM ===
# Su dung native API lgb.train voi so vong co dinh (800 rounds).
# Khong dung early stopping tren T11 de dam bao validation sach.

y_train_log = np.log1p(y_train)
dtrain = lgb.Dataset(X_train, label=y_train_log, categorical_feature=cat_cols, free_raw_data=False)

params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "verbose": -1,
    "seed": SEED,
    "n_jobs": -1,
}

NUM_BOOST_ROUND = 800

model = lgb.train(
    params, dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain],
    valid_names=["train"],
    callbacks=[lgb.log_evaluation(period=100)],
)
print(f"\nHuan luyen xong voi {NUM_BOOST_ROUND} rounds.")


In [ ]:
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 8))
top_n = min(25, len(fi))
plt.barh(fi["feature"].head(top_n).values[::-1], fi["importance"].head(top_n).values[::-1], color="#4C72B0")
plt.title("Top Feature Importance (Gain)", fontsize=14)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
fi.head(20)


## Đánh giá metric trên tháng 11 (Validation)

- `MAPE_quantity = mean(|actual_qty - pred_qty| / actual_qty)` chỉ với actual_qty > 0
- `MAPE_revenue = mean(|actual_rev - pred_rev| / actual_rev)` chỉ với actual_rev > 0
- `pred_revenue = pred_qty * price_ref`


In [ ]:
pred_log = model.predict(X_val, num_iteration=NUM_BOOST_ROUND)
pred_qty = np.expm1(pred_log)
pred_qty = np.clip(pred_qty, 0, None)

pr = val_pd["price_ref"].values if "price_ref" in val_pd.columns else np.ones(len(pred_qty))
pred_rev = pred_qty * np.where(np.isnan(pr) | (pr == 0), 1.0, pr)

actual_qty = y_val.values
actual_rev = y_val_rev.values

mae_qty = mean_absolute_error(actual_qty, pred_qty)
mask_qty = actual_qty > 0
mape_qty = np.mean(np.abs(actual_qty[mask_qty] - pred_qty[mask_qty]) / actual_qty[mask_qty]) if mask_qty.sum() > 0 else np.nan

mae_rev = mean_absolute_error(actual_rev, pred_rev)
mask_rev = actual_rev > 0
mape_rev = np.mean(np.abs(actual_rev[mask_rev] - pred_rev[mask_rev]) / actual_rev[mask_rev]) if mask_rev.sum() > 0 else np.nan

print("=" * 55)
print("       EVALUATION METRICS (Validation: T11)")
print("=" * 55)
metrics_table = pd.DataFrame({
    "MAE_quantity": [mae_qty],
    "MAPE_quantity (metric chinh)": [mape_qty],
    "MAE_revenue": [mae_rev],
    "MAPE_revenue": [mape_rev],
}).T
metrics_table.columns = ["Value"]
print(metrics_table.to_string())
print("=" * 55)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
ax.scatter(actual_qty, pred_qty, alpha=0.15, s=8, color="#2ecc71")
mx = max(actual_qty.max(), pred_qty.max())
ax.plot([0, mx], [0, mx], "r--", lw=1, label="y=x")
ax.set_xlabel("Actual Qty"); ax.set_ylabel("Predicted Qty")
ax.set_title("Actual vs Predicted (Quantity)"); ax.legend()

ax = axes[1]
residuals = actual_qty - pred_qty
ax.hist(residuals, bins=80, color="#3498db", alpha=0.7, edgecolor="white")
ax.axvline(0, color="red", ls="--")
ax.set_title("Phan bo residuals (qty)"); ax.set_xlabel("Actual - Predicted")

ax = axes[2]
ax.hist(actual_qty[actual_qty > 0], bins=80, alpha=0.5, label="Actual", color="#e74c3c")
ax.hist(pred_qty[pred_qty > 0], bins=80, alpha=0.5, label="Predicted", color="#2ecc71")
ax.set_title("Phan bo quantity"); ax.legend(); ax.set_xlabel("Quantity")
plt.tight_layout(); plt.show()


In [ ]:
val_result = pd.DataFrame({
    "location": val_pd["location"].values,
    "item_id": val_pd["item_id"].values,
    "target_month": 11,
    "actual_qty": actual_qty,
    "pred_qty": np.round(pred_qty, 4),
    "actual_revenue": actual_rev,
    "pred_revenue": np.round(pred_rev, 4),
})
val_path = str(OUT / "lightgbm_rolling_validation_predictions.csv")
val_result.to_csv(val_path, index=False)
print(f"Da luu validation predictions: {val_path}")
print(f"   {val_result.shape[0]:,} dong")

metrics_save = pd.DataFrame({
    "metric": ["MAE_quantity","MAPE_quantity","MAE_revenue","MAPE_revenue"],
    "value": [mae_qty, mape_qty, mae_rev, mape_rev],
    "note": ["","metric chinh","",""],
})
met_path = str(OUT / "lightgbm_rolling_metrics.csv")
metrics_save.to_csv(met_path, index=False)
print(f"Da luu metrics: {met_path}")
val_result.head(10)


In [ ]:
print("=" * 60)
print("  TOM TAT KET QUA -- LightGBM Rolling-Sample Forecasting")
print("  Rolling: T1-T6->T7, T1-T7->T8, T1-T8->T9, T1-T9->T10")
print("  Validation: T1-T10 -> T11 (held-out)")
print("=" * 60)
print(f"  So dong huan luyen (rolling)   : {X_train.shape[0]:,}")
print(f"  So dong validation (T11)       : {X_val.shape[0]:,}")
print(f"  So features                    : {X_train.shape[1]}")
print(f"  Boosting rounds                : {NUM_BOOST_ROUND}")
print(f"  -------------------------------------------")
print(f"  MAE  quantity                  : {mae_qty:.4f}")
print(f"  MAPE quantity (MAIN)           : {mape_qty:.4f}")
print(f"  MAE  revenue                   : {mae_rev:.4f}")
print(f"  MAPE revenue                   : {mape_rev:.4f}")
print("=" * 60)
